# Init - Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "martial_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}

# Reading from bronze

In [0]:
df = spark.table('workspace.bronze.crm_custumer_info')

In [0]:
#trim the string
#normalization for martial_status, gndr
#names are not friendly
df.display()

# Data Transformations

## Trimming values

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalization

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status"
        , F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
            .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
            .otherwise("n/a")
    )

    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
            .when(F.upper(F.col("cst_gndr")) == "M", "Male")
            .otherwise("n/a")
    )

)

## Remove records with missing customer ID

In [0]:
df = df.filter(col("cst_id").isNotNull())

## Renaming the columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe

In [0]:
df.limit(10).display()

# Write into Silver Table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite") # it's good for small dataframes 
    .option("overwriteSchema", "true")
    .saveAsTable("silver.crm_customers")
)

In [0]:
%sql
SELECT * 
FROM workspace.silver.crm_customers
LIMIT 10